In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import time
import pickle
from collections import defaultdict
from datasets import load_dataset

# --- Configuration (Optimized for 16GB VRAM) ---
CONFIG = {
    'block_size': 256,
    'batch_size': 8,      # REDUCED: Physical batch size
    'grad_accum_steps': 8, # NEW: Accumulate to reach effective batch of 64
    'n_layer': 48,        # Deep network
    'n_head': 8,
    'n_embd': 256,
    'dropout': 0.1,
    'lr': 3e-4,
    'max_iters': 20000,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# --- Data Preparation ---
def prepare_wikitext():
    print("Loading WikiText-103 (Raw) via HuggingFace Datasets...")
    try:
        dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
        print("Merging text data...")
        text = "\n".join(dataset['text'])
        print(f"Loaded {len(text)} characters.")
        return text
    except Exception as e:
        print(f"Error loading dataset: {e}")
        raise e

class CharDataset(Dataset):
    def __init__(self, text, block_size):
        chars = sorted(list(set(text)))
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.vocab_size = len(chars)
        self.block_size = block_size
        print(f"Vocab size: {self.vocab_size}")
        print("Tokenizing...")
        self.data = np.array([self.stoi[ch] for ch in text], dtype=np.uint16)
    
    def __len__(self):
        return len(self.data) - self.block_size
    
    def __getitem__(self, idx):
        chunk = self.data[idx:idx+self.block_size+1]
        x = torch.from_numpy(chunk[:-1].astype(np.int64))
        y = torch.from_numpy(chunk[1:].astype(np.int64))
        return x, y

# --- Model (Unchanged) ---
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_attn = nn.Linear(config['n_embd'], 3 * config['n_embd'])
        self.c_proj = nn.Linear(config['n_embd'], config['n_embd'])
        self.attn_dropout = nn.Dropout(config['dropout'])
        self.resid_dropout = nn.Dropout(config['dropout'])
        self.n_head = config['n_head']
        self.n_embd = config['n_embd']
        self.register_buffer("bias", torch.tril(torch.ones(config['block_size'], config['block_size']))
                                     .view(1, 1, config['block_size'], config['block_size']))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        
        att = (q @ k.transpose(-2, -1)) * (1.0 / np.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config['n_embd'], 4 * config['n_embd'])
        self.c_proj = nn.Linear(4 * config['n_embd'], config['n_embd'])
        self.dropout = nn.Dropout(config['dropout'])
    def forward(self, x):
        return self.dropout(self.c_proj(F.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config, use_residual=True):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config['n_embd'])
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config['n_embd'])
        self.mlp = MLP(config)
        self.use_residual = use_residual
    def forward(self, x):
        if self.use_residual:
            x = x + self.attn(self.ln_1(x))
            x = x + self.mlp(self.ln_2(x))
        else:
            x = self.attn(self.ln_1(x))
            x = self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config, vocab_size, use_residual=True):
        super().__init__()
        self.block_size = config['block_size']
        self.tok_emb = nn.Embedding(vocab_size, config['n_embd'])
        self.pos_emb = nn.Parameter(torch.zeros(1, config['block_size'], config['n_embd']))
        self.drop = nn.Dropout(config['dropout'])
        self.blocks = nn.ModuleList([Block(config, use_residual) for _ in range(config['n_layer'])])
        self.ln_f = nn.LayerNorm(config['n_embd'])
        self.head = nn.Linear(config['n_embd'], vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        b, t = idx.size()
        tok_emb = self.tok_emb(idx)
        pos_emb = self.pos_emb[:, :t, :]
        x = self.drop(tok_emb + pos_emb)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

def stable_rank(matrix):
    """Compute stable rank: ||A||_F^2 / ||A||_2^2"""
    with torch.no_grad():
        frobenius_norm = torch.norm(matrix, p='fro')
        spectral_norm = torch.linalg.matrix_norm(matrix, ord=2)
        srank = (frobenius_norm ** 2) / (spectral_norm ** 2 + 1e-8)
    return srank.item()

class TimeStopper:
    def __init__(self, limit_hours=11.0):
        self.start_time = time.time()
        self.limit_seconds = limit_hours * 3600
    def should_stop(self):
        return (time.time() - self.start_time) > self.limit_seconds

In [2]:
# [Insert Updated Part 1 Here]

def train_baseline():
    print(f"BASELINE: {CONFIG['n_layer']} Layers WITH Residuals")
    print(f"Batch: {CONFIG['batch_size']} (Accum: {CONFIG['grad_accum_steps']}) -> Eff: {CONFIG['batch_size']*CONFIG['grad_accum_steps']}")
    
    text = prepare_wikitext()
    dataset = CharDataset(text, CONFIG['block_size'])
    dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    
    model = GPT(CONFIG, dataset.vocab_size, use_residual=True).to(CONFIG['device'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'])
    scaler = torch.amp.GradScaler('cuda') # For Mixed Precision
    
    losses = []
    timer = TimeStopper(limit_hours=11.0)
    iter_count = 0
    micro_step = 0
    
    model.train()
    optimizer.zero_grad()
    
    while iter_count < CONFIG['max_iters']:
        for x, y in dataloader:
            if iter_count >= CONFIG['max_iters']: break
            if timer.should_stop(): break
            
            x, y = x.to(CONFIG['device']), y.to(CONFIG['device'])
            
            # Mixed Precision Context
            with torch.amp.autocast('cuda'):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
                loss = loss / CONFIG['grad_accum_steps'] # Scale loss
            
            scaler.scale(loss).backward()
            
            micro_step += 1
            if micro_step % CONFIG['grad_accum_steps'] == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
                losses.append(loss.item() * CONFIG['grad_accum_steps'])
                iter_count += 1
                if iter_count % 100 == 0:
                    print(f"Iter {iter_count}: Loss {losses[-1]:.4f}")
        
        if timer.should_stop(): break

    with open('baseline_results.pkl', 'wb') as f:
        pickle.dump({'losses': losses}, f)
    print("Baseline done.")

if __name__ == "__main__":
    train_baseline()

BASELINE: 48 Layers WITH Residuals
Batch: 8 (Accum: 8) -> Eff: 64
Loading WikiText-103 (Raw) via HuggingFace Datasets...


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Merging text data...
Loaded 540095682 characters.
Vocab size: 4979
Tokenizing...
Iter 100: Loss 2.9881
Iter 200: Loss 2.6750
Iter 300: Loss 2.6878
Iter 400: Loss 2.4971
Iter 500: Loss 2.4778
Iter 600: Loss 2.4175
Iter 700: Loss 2.4129
Iter 800: Loss 2.3462
Iter 900: Loss 2.1931
Iter 1000: Loss 2.1823
Iter 1100: Loss 2.1249
Iter 1200: Loss 2.0438
Iter 1300: Loss 1.9766
Iter 1400: Loss 1.8643
Iter 1500: Loss 1.8829
Iter 1600: Loss 1.7145
Iter 1700: Loss 1.7487
Iter 1800: Loss 1.6229
Iter 1900: Loss 1.5926
Iter 2000: Loss 1.5147
Iter 2100: Loss 1.6880
Iter 2200: Loss 1.5754
Iter 2300: Loss 1.6071
Iter 2400: Loss 3.6314
Iter 2500: Loss 2.3017
Iter 2600: Loss 1.6535
Iter 2700: Loss 1.5320
Iter 2800: Loss 1.5578
Iter 2900: Loss 1.4508
Iter 3000: Loss 1.5040
Iter 3100: Loss 1.4357
Iter 3200: Loss 1.4612
Iter 3300: Loss 1.3832
Iter 3400: Loss 1.3138
Iter 3500: Loss 1.3401
Iter 3600: Loss 1.3153
Iter 3700: Loss 1.3109
Iter 3800: Loss 1.3786
Iter 3900: Loss 1.3659
Iter 4000: Loss 1.3742
Iter 410